# Text Classification with Naive Bayes & TF-IDF
Accompanies **Chapter 11: Naive Bayes Classification**.

Naive Bayes applies Bayes' Theorem with the conditional independence assumption:
$$P(y = c \mid \mathbf{x}) \propto P(y = c) \prod_{j=1}^p P(x_j \mid y = c)$$

For text documents, **Multinomial Naive Bayes** treats each word count as a multinomial draw. When a word in the test document was never observed during training, $P(w_j \mid y=c) = 0$, causing the entire product to collapse to zero (the zero-frequency problem). 
We resolve this via **Laplace Smoothing**:
$$\hat{\theta}_{jc} = \frac{N_{jc} + \alpha}{N_c + \alpha V}$$

In this notebook, we classify financial consumer complaints from `Consumer_Complaints_train.csv.zip`.

In [ ]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

zip_path = os.path.abspath('../../data/Consumer_Complaints_train.csv.zip')
df = pd.read_csv(zip_path, compression='zip')

narrative_col = 'Consumer complaint narrative'
target_col = 'Product'

clean_df = df[[narrative_col, target_col]].dropna().sample(n=6000, random_state=42)
print("Top product categories:")
print(clean_df[target_col].value_counts().head(4))

X_train_raw, X_test_raw, y_train, y_test = train_test_split(clean_df[narrative_col], clean_df[target_col], test_size=0.25, random_state=42)

### TF-IDF Vectorization & Multinomial Estimation
TF-IDF balances term frequency against inverse document frequency:
$$\text{tf-idf}(t, d, D) = \text{tf}(t, d) \times \log\left(\frac{1 + |D|}{1 + |\{d \in D : t \in d\}|}\right)$$
We apply `alpha=1.0` (Laplace smoothing) to prevent zero-probability collapse on unseen terms.

In [ ]:
vectorizer = TfidfVectorizer(max_features=4000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train_raw)
X_test_vec = vectorizer.transform(X_test_raw)

nb = MultinomialNB(alpha=1.0)
nb.fit(X_train_vec, y_train)

y_pred = nb.predict(X_test_vec)
print(classification_report(y_test, y_pred))